# SQUASSSH training example



In [ ]:
# This script can run from either google colab or from within the SQUASSH repository.
# The code needs to be installed from github if running on colab. 
import os
if os.getenv("COLAB_RELEASE_TAG"):
    !git clone --depth 1 https://github.com/edrosten/squassh.git
    import sys
    sys.path.insert(0, '/content/squassh')
    %pip install pystrict plotly

os.environ["OVERRIDE_UNCLEAN_REPO"]="1"

In [ ]:
from typing import cast
import torch
from torch import Tensor
import torch._dynamo
import resi_data   
import mark_bates_data
import train
import train_nupc
import network
import device
from localisation_data import LocalisationDataSetMultipleDan6

## Load in the dataset

Load data and select some rendering parameters to give a useful rendition.

In [ ]:
nupc3d = [t.to(device.device).half() for t in resi_data.load_3d()]

rejection = 1.0
mult = 20 # 

data_parameters = train.DataParametersXYYZ(
    image_size_xy = 64,
    image_size_z = 32,
    nm_per_pixel_xy = 3.9,
    z_scale = 2
)

## First phase: rapid training with a small model

Initial training starts with a small model of 35 points and decreases the rendering resolution from 65 to 34nm and then slowly to 13nm. Since there are so few points, the intensities are fixed.

In [ ]:
model_size=35
net, parameterisation =train_nupc.PredictReconstruction(initial_model_size=model_size, final_model_size=model_size*mult, **vars(data_parameters), data=nupc3d)
net._model_intensities.requires_grad=False  # pylint: disable=protected-access
parameterisation.max_stretch_factor_axis = torch.tensor(2.0)
parameterisation.max_stretch_factor_expand = torch.tensor(1.0)
_ = net.to(device.device)


### Fast training schedule

The schedule starts very coarse and somewhat rapidly decays the blur down to the final value of 13nm

In [ ]:
params_initial = train.TrainingParameters()
params_initial.batch_size = 160
params_initial.validity_weight=rejection

params_initial.schedule[0].epochs = 90
params_initial.schedule[0].initial_psf = 65.0
params_initial.schedule[0].final_psf = 33.8
params_initial.schedule[0].psf_step_every= 30
params_initial.schedule[0].initial_lr= 0.0001
params_initial.schedule[0].final_lr= 0.0001

params_initial.schedule.append(train.TrainingSegment())
params_initial.schedule[1].epochs = 300
params_initial.schedule[1].initial_psf = 24.7
params_initial.schedule[1].final_psf = 13.0
params_initial.schedule[1].psf_step_every= 100
params_initial.schedule[1].initial_lr= 0.0001
params_initial.schedule[1].final_lr= 0.0001

dataset_initial = LocalisationDataSetMultipleDan6(**vars(data_parameters), data=nupc3d, augmentations=8, device=device.device)

Train the model and save the results in a subdirectory called `phase_0`. The complete run is saved in `logs-`*timestamp*`-`*git hash*.

Note that `torch.compile` has a large effect on speed and especially memory consumption for this code, so this won't run well on GPUs older than the 2000 series (it was tested on a 2080Ti). But `torch.compile` has historically been a bit buggy so it's safer to reset the compuler before using it.

In [ ]:
torch.compiler.reset()
fast = cast(network.GeneralPredictReconstruction, torch.compile(net))
train.retrain(fast, dataset_initial, params_initial, 'phase_0')

## Second training phase

This phase first replaces each of the 35 initial points with 20 in roughly the same location, and re-enables learning of the point intensities. 

In [ ]:
scatter = 0.01
scale = net.get_model()[0].abs().max().item()

old_pts, old_weights = (j.detach() for j in net.get_model())

new_pts = torch.nn.functional.interpolate(old_pts.unsqueeze(0).unsqueeze(0), scale_factor=[mult,1]).squeeze(0).squeeze(0)
new_pts += torch.randn(new_pts.shape, device=device.device) * scale * scatter

new_weights = torch.nn.functional.interpolate(old_weights.unsqueeze(0).unsqueeze(0), scale_factor=mult).squeeze(0).squeeze(0)

net.set_model(new_pts, new_weights)
net._model_intensities.requires_grad=True  # pylint: disable=protected-access

At this point, the training will have found the principle axis. So, we need to turn off optimization when we enable expansions along the other axes because if there are three independent scaling axes, then there is no real notion of overall orientation, and the orientation will drift relative to the first axis.  

In [ ]:
parameterisation.principal_axis.requires_grad = False
parameterisation.max_stretch_factor_expand = torch.tensor(1.3, device=device.device)

...then continue to train with a 13nm resolution with a slowly decreasing learning rate.

In [ ]:
params_refine = train.TrainingParameters()
params_refine.batch_size = 10
params_refine.validity_weight=rejection

params_refine.schedule[0].epochs = 500
params_refine.schedule[0].initial_psf = 13.0
params_refine.schedule[0].final_psf = 13.0
params_refine.schedule[0].psf_step_every= 300
params_refine.schedule[0].initial_lr= 0.0002
params_refine.schedule[0].final_lr= 0.00005

torch.compiler.reset() # Otherwise it crashes on torch 2.7
fast = cast(network.GeneralPredictReconstruction, torch.compile(net))

dataset_refine = LocalisationDataSetMultipleDan6(**vars(data_parameters), data=nupc3d, augmentations=1, device=device.device)
train.retrain(fast, dataset_refine, params_refine, 'phase_1')

## Third training phase

For the third training phase we allow the system to predict shifts for each point independently. This is very overparameterised, so for we allow a small shift relative to the existing distortions. For stability we freeze everything except for the small part of the network predicting the shifts.

In [ ]:
parameterisation.shift_amount_nm = torch.tensor(7)
parameterisation.per_point_shift=True
    
# Turn off gradients etc for everything
net.eval()
for p in net.parameters():
    p.requires_grad = False

# Turn gradients etc back on only for the per-point shift
parameterisation.shift_network.train()
for p in parameterisation.shift_network.parameters():
    p.requires_grad = True

Then continue training at the low learning rate at the final blur level

In [ ]:

params_final = train.TrainingParameters()
params_final.batch_size = 10
params_final.validity_weight=rejection
params_final.checkpoint_every=100

params_final.schedule[0].epochs = 500
params_final.schedule[0].initial_psf = 13
params_final.schedule[0].final_psf = 13
params_final.schedule[0].psf_step_every= 300
params_final.schedule[0].initial_lr= 0.00005
params_final.schedule[0].final_lr= 0.00005

torch.compiler.reset()
fast = cast(network.GeneralPredictReconstruction, torch.compile(net))
train.retrain(fast, dataset_refine, params_final, 'phase_2')


Now freeze the network

In [ ]:
net=net.eval()
for i in net.parameters():
    i.requires_grad=False


## Plot the learned 3D model and stretch axis

Plot an XY projection, along with the axis of stretch. Note that overall orientation of the model is effectively random.


Create a mesh from the model. Note mesh creation is very GPU RAM intensive, so since it's a one off the most hassle free solution is simply run it on the CPU.

In [ ]:
from save_ply import make_mesh
v,f = make_mesh(*[i.detach().cpu() for i in net.get_model()], 2.0, size=100)

Plot the mesh and main stretch axis in 3D

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
import sys
pio.renderers.default = 'colab' if 'google.colab' in sys.modules else 'notebook'
ax = parameterisation.get_axis().cpu().detach()
ax = torch.stack([ax*50, ax*-50], 0)

fig=go.Figure(go.Mesh3d(
    x=v[:,0], y=v[:,1], z=v[:,2],
    i=f[:,0], j=f[:,1], k=f[:,2]
))
fig.add_traces([
    go.Scatter3d(x=ax[:,0], y=ax[:,1], z=ax[:,2], line={"color":"red", "width":8}, marker={"size":0})
])
fig.show()

# Analyze the results using PCA

First, run all the data through the network and record the point positions after the parameterisation has been applied but before the final Euclidean transformation. Only keep the point position where the network indicates that the output is valid. The majority of validities are very close to 1 or 0, so this is generally insensitive to the particular value of the threshold.

In this case we we keep the output of the parameterisation. These points are all in the space of the underlying model, i.e. they have had the parameterisation applied but have not yet been rotated and translated to fit the image. We want these points, because for the PCA analysis, the final rotation and translation are not interesting changes and will contaminate the interesting changes found by PCA.

In [ ]:
import tqdm
from torch.utils.data import DataLoader

final_fwhm=13
final_sigma_t = torch.tensor(train.fwhm_to_sigma(final_fwhm), device=device.device)
loader = DataLoader(dataset_refine, batch_size=1, shuffle=False)

def apply_net_to_data(loader: DataLoader)->torch.Tensor:
    pts_list = []

    for index,datum in enumerate(tqdm.tqdm(loader)):
        _,_,_,is_valid,parameters = net.process_input(datum, min_sigma_nm=final_sigma_t)
        points, _ , _ = parameterisation(*net.get_model(), parameters)
    
        if is_valid > 0.5:
            pts_list.append(points.cpu().squeeze(0))
    return torch.stack(pts_list, 0)

results_pts = apply_net_to_data(loader)

### Compute PCA of the point positions using the singular value decomposition.

Point positions (700 3D points in this case) are treated as a 1-D vector of length 2100. Given the SVD as $U\  \text{diag}(S) V^T$, the components are the rows of $V$.


In [ ]:
import math
from dataclasses import dataclass
@dataclass
class _PCAResult:
    S: Tensor
    Vh: Tensor
    stddev: Tensor
    centre: Tensor


def _PCA(points:Tensor)->_PCAResult:
    n_data = points.shape[0]
    flat_pts =points.reshape(n_data, -1)

    flat_pts_centred = flat_pts - flat_pts.mean(0).unsqueeze(0).expand(n_data, -1)
    (_, S, Vh_vectors) = torch.linalg.svd(flat_pts_centred, full_matrices=False) # pylint: disable=not-callable

    # Covariances are S^2 / (n-1)
    # standard devs are S/sqrt(n-1)
    stddev = S / (math.sqrt(n_data-1))
    centre = flat_pts.mean(0).reshape(-1, 3)
    Vh_vectors = Vh_vectors.reshape(Vh_vectors.shape[0], *centre.shape)

    return _PCAResult(S=S,Vh=Vh_vectors,stddev=stddev,centre=centre)




### Plot the first 3 PCA components 

The mean is given in black, the component is given in orange. Sinc PCA is symmetric, and the motions are small we plot only at +3σ.

In [ ]:
import matplotlib.pyplot as plt
import matrix
pca = _PCA(results_pts)
centre = pca.centre
Vh = pca.Vh
centre = pca.centre
stddev = pca.stddev

# Reorder the points so that the darkest (i.e. closest to black in the data
# which is closest to white here) are drawn first with scatter(). This means
# that a high brigtness point won't be obscured by a very dim one, so scatter 
# gives a better approximation of a proper rendering. 
intensities = net.get_model()[1].cpu().detach()
_, darkest_first = intensities.sort()
intensities = intensities[darkest_first]
centre = centre[darkest_first,:]
Vh = Vh[:, darkest_first, :]

# The system learns the stretch axis, i.e. the axis aligned with the centre of the two 
# rings as the X axis of R. Therefore for display, rotate it so that the stretch axis 
# is aligned with Z instead. 
R = matrix.euler(90*torch.tensor([torch.pi])/180, 'y').squeeze() @ parameterisation.get_R().cpu()

# Segment the rings from the data as the points with positive and negative Z.
top_mask = (R @ centre.permute(1,0)).permute(1,0)[:,2] > 0

N=3 
alpha=0.1
plt.clf()
for I in range(3):
    component = Vh[I]*stddev[I]*3

    plt.subplot(2,N,I+1)
    plt.scatter(*(R @ (centre          )[top_mask,:].permute(1,0))[0:2,:], c=intensities[top_mask], alpha=alpha, cmap='Greys', edgecolors='none')  # type: ignore[misc]
    plt.scatter(*(R @ (centre+component)[top_mask,:].permute(1,0))[0:2,:], c=intensities[top_mask], alpha=alpha, cmap='Oranges', edgecolors='none')  # type: ignore[misc]
    plt.xlabel(f'Component {I+1}')
    plt.axis('square')
    plt.axis((-65,65,-65,65))
    for line in ['top', 'bottom', 'left', 'right']:
        plt.gca().spines[line].set_visible(False)
    plt.gca().set_xticks([])
    plt.gca().set_yticks([])
    plt.gca().xaxis.set_label_position('top')
    if I == 0:
        plt.ylabel('Upper ring')

    plt.subplot(2,N,I+1+N)
    plt.scatter(*(R @ (centre          )[top_mask.logical_not(),:].permute(1,0))[0:2,:], c=intensities[top_mask.logical_not()], alpha=alpha, cmap='Greys', edgecolors='none')  # type: ignore[misc]
    plt.scatter(*(R @ (centre+component)[top_mask.logical_not(),:].permute(1,0))[0:2,:], c=intensities[top_mask.logical_not()], alpha=alpha, cmap='Oranges', edgecolors='none')  # type: ignore[misc]
    plt.axis('square')
    plt.axis((-65,65,-65,65))
    for line in ['top', 'bottom', 'left', 'right']:
        plt.gca().spines[line].set_visible(False)
    plt.gca().set_xticks([])
    plt.gca().set_yticks([])
    if I == 0:
        plt.ylabel('Lower ring')
plt.tight_layout()
plt.pause(.1)

